In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import LogLocator, LogFormatterSciNotation
import country_converter as coco
import time
import warnings
from matplotlib import rcParams
rcParams['font.family'] = 'serif'
rcParams['font.serif'] = ['Times New Roman']
warnings.filterwarnings("ignore")

start_time = time.time()

# ==========================================
# 1. 读取地图与极速轻量化处理
# ==========================================
print("正在读取地图...")
world = gpd.read_file(r"D:\A.baumannii\scripts\worldmap\standard_world_map-main\std_worldmap.shp")
world["geometry"] = world["geometry"].simplify(tolerance=20000, preserve_topology=False)

unique_admins = world["admin"].dropna().unique()
iso_dict = dict(zip(unique_admins, coco.convert(names=unique_admins.tolist(), to="ISO3")))
world["iso_a3"] = world["admin"].map(iso_dict).replace("not found", pd.NA)
world["iso_a3"] = world["iso_a3"].astype(str)


# ==========================================
# 2. 读取 CSV 并按国家统计样本数量
# ==========================================
print("正在处理样本数据...")
df_raw = pd.read_csv(r"D:\A.baumannii\data\metadata.csv")

df_valid = df_raw.dropna(subset=["country"])
df_counts = df_valid['country'].value_counts().reset_index()
df_counts.columns = ['country', 'sample_count']

unique_countries = df_counts["country"].unique()
df_iso_dict = dict(zip(unique_countries, coco.convert(names=unique_countries.tolist(), to="ISO3")))
df_counts["iso_a3"] = df_counts["country"].map(df_iso_dict).replace("not found", pd.NA)
df_counts = df_counts.dropna(subset=["iso_a3"])
df_counts["iso_a3"] = df_counts["iso_a3"].astype(str)


# ==========================================
# 3. 合并数据
# ==========================================
print("正在合并地理与统计数据...")
merged_world = world.merge(df_counts, on="iso_a3", how="left")


# ==========================================
# 4. 绘图与样式定制
# ==========================================
print("正在渲染高精度对数地图...")

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

fig, ax = plt.subplots(1, 1, figsize=(16, 8))

# 画底图
merged_world.plot(ax=ax, color="lightgrey", edgecolor="black", linewidth=0.2)

# 渐变色
colors = ["#74add1", "#ffffbf", "#f46d43", "#a50026"] 
custom_cmap = mcolors.LinearSegmentedColormap.from_list("BlueRed", colors)

data_to_plot = merged_world.dropna(subset=['sample_count']).copy()
data_to_plot = data_to_plot[data_to_plot['sample_count'] > 0]

vmin = max(1, data_to_plot['sample_count'].min())
vmax = data_to_plot['sample_count'].max()
norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)

# 画有数据的国家
data_to_plot.plot(
    column='sample_count', 
    ax=ax,
    cmap=custom_cmap,
    norm=norm,             
    edgecolor="black", 
    linewidth=0.2
)

ax.set_axis_off()


# ==========================================
# 5. 【修改部分】配置左下角竖向图例，标题置顶
# ==========================================
cbar_ax = fig.add_axes([0.15, 0.15, 0.015, 0.35]) 
sm = plt.cm.ScalarMappable(cmap=custom_cmap, norm=norm)
sm._A = []

# 竖向图例
cbar = fig.colorbar(sm, cax=cbar_ax, orientation='vertical')

# 🚨 使用 set_title 将文字强制放在色条正上方
# pad=15 是为了让文字和色带之间留出一点呼吸空间，不至于贴得太紧
cbar.ax.set_title('Isolate count\n(log scale)', fontsize=14, pad=15)
cbar.ax.tick_params(labelsize=12)

# 竖向图例的刻度在 Y 轴上
cbar.ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=5))
cbar.ax.yaxis.set_major_formatter(LogFormatterSciNotation(base=10.0))


# ==========================================
# 6. 纯矢量输出
# ==========================================
plt.savefig("world_map_result.pdf", format="pdf", bbox_inches="tight", pad_inches=0.2)
plt.close()

end_time = time.time()
print(f"🎉 全部完成！耗时：{end_time - start_time:.2f} 秒。图例标题置顶版地图已保存！")

Akrotiri Sovereign Base Area not found in regex
Baykonur Cosmodrome not found in regex
Bajo Nuevo Bank (Petrel Is.) not found in regex
Clipperton Island not found in regex
Coral Sea Islands not found in regex
Cura?ao not found in regex
Dhekelia Sovereign Base Area not found in regex
Indian Ocean Territories not found in regex
Scarborough Reef not found in regex


正在读取地图...


Serranilla Bank not found in regex
Siachen Glacier not found in regex
Spratly Islands not found in regex
US Naval Base Guantanamo Bay not found in regex
Africa not found in regex


正在处理样本数据...
正在合并地理与统计数据...
正在渲染高精度对数地图...
🎉 全部完成！耗时：1.31 秒。图例标题置顶版地图已保存！
